In [ ]:
import numpy as np
from glob import glob

# load filenames for human and dog images
human_files = np.array(glob("/data/lfw/*/*"))
dog_files = np.array(glob("/data/dog_images/*/*/*"))

# print number of images in each dataset
print('There are %d total human images.' % len(human_files))
print('There are %d total dog images.' % len(dog_files))

In [ ]:
import cv2                
import matplotlib.pyplot as plt                        
%matplotlib inline                               

# extract pre-trained face detector
face_cascade = cv2.CascadeClassifier('haarcascades/haarcascade_frontalface_alt.xml')

# load color (BGR) image
img = cv2.imread(human_files[0])
# convert BGR image to grayscale
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# find faces in image
faces = face_cascade.detectMultiScale(gray)

# print number of faces detected in the image
print('Number of faces detected:', len(faces))

# get bounding box for each detected face
for (x,y,w,h) in faces:
    # add bounding box to color image
    cv2.rectangle(img,(x,y),(x+w,y+h),(255,0,0),2)
    
# convert BGR image to RGB for plotting
cv_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# display the image, along with bounding box
plt.imshow(cv_rgb)
plt.show()

In [ ]:
# returns "True" if face is detected in image stored at img_path
def face_detector(img_path):
    img = cv2.imread(img_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray)
    return len(faces) > 0

In [ ]:
from tqdm import tqdm

human_files_short = human_files[:100]
dog_files_short = dog_files[:100]

## Test the performance of the face_detector algorithm 
## on the images in human_files_short and dog_files_short.
detected_humans_human_files = 0
detected_humans_dog_files = 0
for human in range(len(human_files_short)):
    detected_humans_human_files += face_detector(human_files_short[human])
    detected_humans_dog_files += face_detector(dog_files_short[human])
    human += 1
human_percentage_human_files = (detected_humans_human_files/len(human_files_short))*100
human_percentage_dog_files = (detected_humans_dog_files/len(dog_files_short))*100
print("Number of humans detected on " + str(len(human_files_short)) +" human images: " + str(detected_humans_human_files))
print("In human_files, the human faces were detected " + str(human_percentage_human_files) + "% of the time")
print("Number of humans detected on " + str(len(dog_files_short)) +" dog images: " + str(detected_humans_dog_files))
print("In dog_files, the human faces were falsely detected in dog images " + str(human_percentage_dog_files) + "% of the time")

In [ ]:
import torch
import torchvision.models as models

# check if CUDA is available
use_cuda = torch.cuda.is_available()

# define VGG16 model
VGG16 = models.vgg16(pretrained=True)

# move model to GPU if CUDA is available
if use_cuda:
    VGG16 = VGG16.cuda()

In [ ]:
torch.cuda.is_available()

In [ ]:
from PIL import Image
import torchvision.transforms as transforms
import torch

import torchvision
from torchvision import datasets, models, transforms

def load_image(img_path):    
    image = Image.open(img_path).convert('RGB')
    in_transform = transforms.Compose([
                        transforms.Resize(size=(244, 244)),
                        transforms.ToTensor()]) 

    image = in_transform(image)[:3,:,:].unsqueeze(0)
    return image

VGG16.eval() 
def VGG16_predict(img_path):
    '''
    Use pre-trained VGG-16 model to obtain index corresponding to 
    predicted ImageNet class for image at specified path
    
    Args:
        img_path: path to an image
        
    Returns:
        Index corresponding to VGG-16 model's prediction
    '''
    
    ## TODO: Complete the function.
    ## Load and pre-process an image from the given img_path
    ## Return the *index* of the predicted class for that image


    transform = transforms.Compose([            #[1]
    transforms.Resize(256),                    #[2]
    transforms.CenterCrop(224),                #[3]
    transforms.ToTensor(),                     #[4]
    transforms.Normalize(                      #[5]
    mean=[0.485, 0.456, 0.406],                #[6]
    std=[0.229, 0.224, 0.225]                  #[7]
    )])
    img = Image.open(img_path)
    img_t = transform(img)
    batch_t = torch.unsqueeze(img_t, 0)
    model = VGG16.cpu()
    VGG16.eval()
    out = VGG16(batch_t)
    _, index = torch.max(out, 1)
    return index # predicted class index
    

In [ ]:
### returns "True" if a dog is detected in the image stored at img_path
def dog_detector(img_path):
    ## TODO: Complete the function.
    #img = cv2.imread(img_path)
    #gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    dogs = VGG16_predict(img_path)
    if (dogs >= 151 and dogs <= 268):
        return True
    else: 
        return False
    
    #return True # true/false
print(dog_detector(dog_files_short[0]))
print(dog_detector(human_files_short[0]))

In [ ]:
### Test the performance of the dog_detector function
### on the images in human_files_short and dog_files_short.

detected_dogs_dog_files = 0
detected_dogs_human_files = 0
for dog in range(len(dog_files_short)):
    detected_dogs_dog_files += dog_detector(dog_files_short[dog])
    detected_dogs_human_files += dog_detector(human_files_short[dog])
    human += 1
dog_percentage_dog_files = (detected_dogs_dog_files/len(dog_files_short))*100
dog_percentage_human_files = (detected_dogs_human_files/len(human_files_short))*100
print("Number of dogs detected on " + str(len(dog_files_short)) +" dog images: " + str(detected_dogs_dog_files))
print("In dog_files_short, the dogs were correctly detected " + str(dog_percentage_dog_files) + "% of the time")
print("Number of dogs detected on " + str(len(human_files_short)) +" human images: " + str(detected_dogs_human_files))
print("In human_files_short, the dogs were falsely detected in human images " + str(dog_percentage_human_files) + "% of the time")

In [ ]:
import os
from torchvision import datasets
import torchvision.transforms as transforms
import torch
import numpy as np
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

### Write data loaders for training, validation, and test sets
## Specify appropriate transforms, and batch_sizes

batch_size = 20
num_workers = 2

data_dir = '/data/dog_images'
train_dir = os.path.join(data_dir, 'train/')
valid_dir = os.path.join(data_dir, 'valid/')
test_dir = os.path.join(data_dir, 'test/')


standard_normalization = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                              std=[0.229, 0.224, 0.225])

#standard_normalization = transforms.Normalize(mean=[0.5, 0.5, 0.5],
#                                             std=[0.5, 0.5, 0.5])
                                              
data_transforms = {'train': transforms.Compose([#transforms.RandomResizedCrop(224),
                                     transforms.Resize(256),
                                     transforms.RandomResizedCrop(224),
                                     transforms.RandomHorizontalFlip(),
                                     transforms.RandomRotation(30),
                                     transforms.ToTensor(),
                                     standard_normalization]),
                   'val': transforms.Compose([transforms.Resize(size=(224,224)),
                                     transforms.ToTensor(),
                                     standard_normalization]),
                   'test': transforms.Compose([transforms.Resize(size=(224,224)),
                                     transforms.ToTensor(), 
                                     standard_normalization])
                  }


train_data = datasets.ImageFolder(train_dir, transform=data_transforms['train'])
valid_data = datasets.ImageFolder(valid_dir, transform=data_transforms['val'])
test_data = datasets.ImageFolder(test_dir, transform=data_transforms['test'])

train_loader = torch.utils.data.DataLoader(train_data,
                                           batch_size=batch_size, 
                                           num_workers=num_workers,
                                           shuffle=True)
valid_loader = torch.utils.data.DataLoader(valid_data,
                                           batch_size=batch_size, 
                                           num_workers=num_workers,
                                           shuffle=False)
test_loader = torch.utils.data.DataLoader(test_data,
                                           batch_size=batch_size, 
                                           num_workers=num_workers,
                                           shuffle=False)
loaders_scratch = {
    'train': train_loader,
    'valid': valid_loader,
    'test': test_loader
}

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# define the CNN architecture
class Net(nn.Module):
    ### choose an architecture, and complete the class
    def __init__(self):
        super(Net, self).__init__()
        ## Define layers of a CNN
         # convolutional layer
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3,  padding=1)
        self.conv3 = nn.Conv2d(32, 64, 3,  padding=1)
        self.conv4 = nn.Conv2d(64, 128, 3, padding=1)
        # max pooling layer
        self.pool = nn.MaxPool2d(2, 2)
        # fully connected layers
        self.fc1 = nn.Linear(128 * 14 * 14 , 512)
       
        self.fc2 = nn.Linear(512, 133)
        self.dropout = nn.Dropout(.25)
    
    def forward(self, x):
        ## Define forward behavior
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = F.relu(self.conv3(x))
        x = self.pool(x)
        x = F.relu(self.conv4(x))
        x = self.pool(x)

        x = x.view(-1, 128 * 14 * 14)
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# instantiate the CNN
model_scratch = Net()

# move tensors to GPU if CUDA is available
if use_cuda:
    model_scratch.cuda()
    
print(model_scratch)

In [ ]:
import torch.optim as optim

### select loss function
criterion_scratch = nn.CrossEntropyLoss()

### select optimizer
optimizer_scratch = optim.SGD(model_scratch.parameters(), lr = 0.1)

In [ ]:
import numpy as np
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

def train(n_epochs, loaders, model, optimizer, criterion, use_cuda, save_path):
    """returns trained model"""
    # initialize tracker for minimum validation loss
    valid_loss_min = np.Inf 
    
    for epoch in range(1, n_epochs+1):
        # initialize variables to monitor training and validation loss
        running_loss = 0.0
        train_loss = 0.0
        valid_loss = 0.0
        
        ###################
        # train the model #
        ###################
        model.train()
        for batch_idx, (data, target) in enumerate(loaders['train']):
            optimizer.zero_grad()
            # move to GPU
            if use_cuda:
                data, target = data.cuda(), target.cuda()
            ## find the loss and update the model parameters accordingly
            ## record the average training loss, using something like
            optimizer.zero_grad()
            
            output = model(data)
            
            # calculate loss
            loss = criterion(output, target)
            
            # back prop
            loss.backward()
            
            # grad
            optimizer.step()
            
            train_loss = train_loss + ((1 / (batch_idx + 1)) * (loss.data - train_loss))
            
            
            
        ######################    
        # validate the model #
        ######################
        model.eval()
        for batch_idx, (data, target) in enumerate(loaders['valid']):
            # move to GPU
            if use_cuda:
                data, target = data.cuda(), target.cuda()
            
            output = model(data)

            loss = criterion(output, target)

            #valid_loss = valid_loss + ((1 / (batch_idx + 1)) * (loss.data - valid_loss))
            valid_loss = valid_loss + ((1 / (batch_idx + 1)) * (loss.data - valid_loss))

          
        
        train_loss = train_loss/len(loaders['train'].dataset)
        valid_loss = valid_loss/len(loaders['valid'].dataset)
        # print training/validation statistics 
        print('Epoch: {} \tTraining Loss: {:.6f} \tValidation Loss: {:.6f}'.format(
            epoch, 
            train_loss,
            valid_loss
            ))
        
        ## save the model if validation loss has decreased
        if valid_loss < valid_loss_min:

            valid_loss_min = valid_loss

            torch.save(model.state_dict(), save_path) 
            print('Validation Loss improved, saving model....')
       
            
    # return trained model
    return model


# train the model
model_scratch = train(30, loaders_scratch, model_scratch, optimizer_scratch, 
                      criterion_scratch, use_cuda, 'model_scratch.pt')

# load the model that got the best validation accuracy
model_scratch.load_state_dict(torch.load('model_scratch.pt'))

In [ ]:
def test(loaders, model, criterion, use_cuda):

    # monitor test loss and accuracy
    test_loss = 0.
    correct = 0.
    total = 0.

    model.eval()
    for batch_idx, (data, target) in enumerate(loaders['test']):
        # move to GPU
        if use_cuda:
            data, target = data.cuda(), target.cuda()
        # forward pass: compute predicted outputs by passing inputs to the model
        output = model(data)
        # calculate the loss
        loss = criterion(output, target)
        # update average test loss 
        test_loss = test_loss + ((1 / (batch_idx + 1)) * (loss.data - test_loss))
        # convert output probabilities to predicted class
        pred = output.data.max(1, keepdim=True)[1]
        # compare predictions to true label
        correct += np.sum(np.squeeze(pred.eq(target.data.view_as(pred))).cpu().numpy())
        total += data.size(0)
            
    print('Test Loss: {:.6f}\n'.format(test_loss))

    print('\nTest Accuracy: %2d%% (%2d/%2d)' % (
        100. * correct / total, correct, total))

# call test function    
test(loaders_scratch, model_scratch, criterion_scratch, use_cuda)

In [ ]:
## Specify data loaders

loaders_transfer = {
    'train': train_loader,
    'valid': valid_loader,
    'test': test_loader
}

In [ ]:
import torchvision.models as models
import torch.nn as nn

## Specify model architecture 
# model_transfer = VGG16
model_transfer =  models.vgg16(pretrained=True)
# Freeze training for all "features" layers

for param in model_transfer.features.parameters():
    param.requires_grad = False
    
model_transfer.classifier[6].out_features = 133

fc_parameters = model_transfer.classifier[6].parameters()

#Retrain the last fully connected layer
for param in model_transfer.classifier[6].parameters():
    param.requires_grad = True

if use_cuda:
    model_transfer = model_transfer.cuda()
print(model_transfer)

In [ ]:
criterion_transfer = nn.CrossEntropyLoss()

optimizer_transfer = optim.SGD(model_transfer.classifier[6].parameters(), lr=0.001)

In [ ]:
# train the model
n_epochs = 20
model_transfer =  train(n_epochs, loaders_transfer, model_transfer, optimizer_transfer, criterion_transfer, use_cuda, 'model_transfer.pt')

# load the model that got the best validation accuracy (uncomment the line below)
model_transfer.load_state_dict(torch.load('model_transfer.pt'))

In [ ]:
test(loaders_transfer, model_transfer, criterion_transfer, use_cuda)

In [ ]:
### Write a function that takes a path to an image as input
### and returns the dog breed that is predicted by the model.
# list of class names by index, i.e. a name can be accessed like class_names[0]
from keras.preprocessing import image

#class_names = [item[4:].replace("_", " ") for item in loaders_transfer['train'].dataset.classes]
#class_names = loaders_transfer['train'].dataset.classes
class_names = [item[4:] for item in loaders_transfer['train'].dataset.classes]
#class_names_clean = [item.replace("_", " ") for item in class_names]

from torchvision import transforms
transform = transforms.Compose([            #[1]
 transforms.Resize(256),                    #[2]
 transforms.CenterCrop(224),                #[3]
 transforms.ToTensor(),                     #[4]
 transforms.Normalize(                      #[5]
 mean=[0.485, 0.456, 0.406],                #[6]
 std=[0.229, 0.224, 0.225]                  #[7]
 )])

def show_image(path):
    img = image.load_img(path, target_size=(224, 224))
    img = image.img_to_array(img)
    plt.imshow(img/255)
    plt.show()

def predict_breed_transfer(model, classes, path):
    # load the image and return the predicted breed
    img = Image.open(path)
    img_t = transform(img)
    batch_t = torch.unsqueeze(img_t, 0)

    model = model.cpu()
    model.eval()
    out = model(batch_t)[:,:132]

    _, index = torch.max(out[:132], 1)
    percentage = torch.nn.functional.softmax(out[:132], dim=1)[0] * 100
 
    return classes[index]

In [ ]:
predict_breed_transfer(model_transfer, class_names, '002.jpg')

In [ ]:
### Write your algorithm.

def run_app(img_path):
    ## handle cases for a human face, dog, and neither
    img = Image.open(img_path)
    plt.imshow(img)
    plt.show()
    if dog_detector(img_path) is True:
        prediction = predict_breed_transfer(model_transfer, class_names, img_path)
        prediction_clean = prediction.replace("_", " ")
        print("Hello, dog!\nYour predicted breed is... {0}".format(prediction_clean))  
    elif face_detector(img_path) > 0:
        prediction = predict_breed_transfer(model_transfer, class_names, img_path)
        prediction_clean = prediction.replace("_", " ")
        print("Hello, human!\nYou look like a... {0}".format(prediction_clean))
        count = 3
        
        for item in range (len(dog_files)):
            if (prediction in dog_files[item]):
                show_image(dog_files[item])
                count -=1
                if count <=0:
                    
                    break
                            
    else:
        print("Oops... Cannot identify the picture.\nThe image contains something other than a human or a dog.")

In [ ]:
run_app('005.jpg')